# Task 2.1 — Prompt Chaining & Question Decomposition

This notebook implements an advanced **Prompt Chaining** pipeline to address *multi-barrelled questions*, one of the main limitations highlighted in the literature on automatic political discourse analysis.

When a journalist asks multiple questions within the same turn, language models can struggle to ground the political answer correctly. If a politician clearly answers one part of the question but ignores the others, a standard approach may label the entire answer as `Explicit` or `Dodging`, losing important nuance.

To address this limitation, we use **Llama 3.1 8B-Instruct** (previously fine-tuned in Task 2 to recognize the 9 evasion techniques) as the logical engine within a 3-step architecture:

1. **Step 1 (Splitter):** The model analyzes the journalist's original question and, when it detects multiple interrogatives, decomposes it into atomic sub-questions (sQA) returned in JSON format.
2. **Step 2 (Evaluator):** The model evaluates the politician's full answer against each generated sub-question and assigns one of the 9 evasion techniques to each sub-question (e.g. *Explicit*, *Deflection*, *Dodging*).
3. **Step 3 (Aggregator):** A deterministic algorithm analyzes the multiple evaluations. If the politician answered at least one sub-question explicitly but evaded the others, it assigns the aggregate label **Partial/half-answer**, then maps the result to the 3 clarity macro-categories.

*Note: This notebook does not perform training; it runs complex inference by loading the LoRA adapter weights saved at the end of Task 2.*

## Environment setup

Import dependencies, configure paths, and detect the execution environment.


In [ ]:
# !pip install -q -U transformers peft trl bitsandbytes datasets accelerate pyarrow

import os, sys, json, gc, re
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_from_disk
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

try:
    import yaml
except ImportError as exc:
    raise ImportError("Missing dependency: PyYAML. Install with `pip install pyyaml`.") from exc

# Path configuration and environment detection (Colab / Local)
try:
    import google.colab
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)

    # Base paths on Google Drive
    BASE_DIR = "/content/drive/MyDrive/progettoLLM"
    REPO_DIR = os.path.join(BASE_DIR, "CLARITY")

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # HF login through Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Google Colab environment configured successfully.")

except ImportError:
    # Local base paths
    BASE_DIR = ".."
    REPO_DIR = BASE_DIR
    
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Read token from local .env file
    env_path = os.path.join(REPO_DIR, ".env")
    if not os.path.exists(env_path):
        env_path = ".env"
        
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    os.environ['HF_TOKEN'] = hf_token
                    login(token=hf_token)
                    break
    print("Local environment detected.")

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## Configuration, paths, and prompts

Load the Task 2 adapter path and define the splitter/evaluator prompts.


In [ ]:
# Config from file
CONFIG_PATH = os.path.join(REPO_DIR, "config", "config.yml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f) or {}

# Model
MODEL_ID = config.get("model", {}).get("id", "meta-llama/Llama-3.2-3B-Instruct")

# Dataset/augmentation config (only paths are used here)
DATASET_CONFIG = config.get("dataset", {})

# Load Task 2 trained weights by building the path dynamically
RES_DIR = os.path.join(REPO_DIR, config.get("paths", {}).get("results_dir", "results"))

TASK2_DIR = os.path.join(REPO_DIR, config.get("paths", {}).get("task2_dir", "results/task2"))
TASK2_MODEL_PATH = os.path.join(TASK2_DIR, "final_model")
if not os.path.exists(TASK2_MODEL_PATH):
    TASK2_MODEL_PATH = os.path.join(TASK2_DIR, "modello_finale")

# Output folder for Prompt Chaining results
CHAINING_DIR = os.path.join(TASK2_DIR, "chaining")

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# PROMPT 1: Splitter — less conservative version with few-shot examples
PROMPT_SPLITTER = """You are an expert linguistic assistant. Analyze the following question.
Split it into atomic sub-questions when it contains multiple interrogatives, even if they are connected by 'and', 'and do you', 'and how', 'and if', 'also', or by multiple question marks.
Keep a single question only when the text clearly asks for one piece of information.
Do not invent new questions: preserve the original meaning and remove only introductions or polite formulas.

Example 1:
Question: Why was that necessary? And how do you punish a country that is already isolated?
Output: {\"sub_questions\": [\"Why was that necessary?\", \"How do you punish a country that is already isolated?\"]}

Example 2:
Question: Do you agree with him, sir? And do you think that kind of rhetoric is appropriate?
Output: {\"sub_questions\": [\"Do you agree with him, sir?\", \"Do you think that kind of rhetoric is appropriate?\"]}

JSON FORMAT ONLY: {\"sub_questions\": [\"question 1\", \"question 2\"]}
Do not add any other text.
"""

# PROMPT 2: Evaluator
PROMPT_EVALUATOR = """You are an expert political communication analyst. Classify the politician's answer to the question by choosing EXACTLY ONE of the following 9 categories:

1. 'Explicit'
2. 'Declining to answer'
3. 'Claims ignorance'
4. 'Dodging'
5. 'Deflection'
6. 'General'
7. 'Implicit'
8. 'Partial/half-answer'
9. 'Clarification'

Classification examples (few-shot):
Q: "Will you support the new tax law?"
A: "I am not aware of the details of the law." -> {"category": "Claims ignorance"}

Q: "What do you think about your party's scandal?"
A: "You should look at what the opposition did last year!" -> {"category": "Deflection"}

Q: "What measures will you take on climate?"
A: "Climate is important, and we will do our best for everyone." -> {"category": "General"}

Additional guidelines:
- Use 'Explicit' when the answer directly or substantively addresses the question, including through explanations, policy details, numbers, decisions, or a clear position.
- Do not use 'General', 'Deflection', or 'Dodging' if the answer covers the main requested information.
- Use 'Declining to answer' when the speaker explicitly refuses to answer, comment, or discuss the issue.
- Use 'Claims ignorance' when the speaker says they do not know, do not remember, or lack enough information.

Reply ONLY with a valid JSON object containing the single key "category". Do not add markdown formatting, preambles, or explanations.
"""

# Mapping to the 3 macro-categories
MAPPING_9_CLASSES = {
    "Explicit": "Clear Reply",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Dodging": "Ambivalent",
    "Deflection": "Ambivalent",
    "General": "Ambivalent",
    "Implicit": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Clarification": "Ambivalent"
}

print("Paths and prompts loaded.")


## Dataset loading

Load the test split used by the prompt-chaining pipeline.


In [ ]:
# This task only needs the test set for evaluation
# Build the path from REPO_DIR
test_dataset_path = os.path.join(REPO_DIR, DATASET_CONFIG.get("test_path", "dataset/QEvasion/test"))

test_dataset = load_from_disk(test_dataset_path)

print(f"Test set loaded: {len(test_dataset)} examples.")


## Fine-tuned model loading

Load the base model in 4-bit mode and attach the Task 2 LoRA adapter.


In [ ]:
torch.cuda.empty_cache()
gc.collect()

# Heuristic: leave some headroom on GPU and allow CPU offload if needed
if torch.cuda.is_available():
    total_gb = int(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3))
    gpu_gb = max(total_gb - 2, 1)
    MAX_MEMORY = {0: f"{gpu_gb}GiB", "cpu": "64GiB"}
else:
    MAX_MEMORY = {"cpu": "64GiB"}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    llm_int8_enable_fp32_cpu_offload=True,
)

print(f"Loading base model ({MODEL_ID}) in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    max_memory=MAX_MEMORY,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)

print(f"Applying Task 2 adapters from: {TASK2_MODEL_PATH}")
model = PeftModel.from_pretrained(
    base_model, TASK2_MODEL_PATH,
    low_cpu_mem_usage=True, device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(TASK2_MODEL_PATH)
model.eval()

print(f"Inference engine ready! Device: {model.device}")


## Prompt-chaining pipeline

Split questions, evaluate each sub-question, and aggregate the final clarity prediction.


In [ ]:
def generate_llm_output(system_prompt, user_content, max_tokens=150, use_base_model=False):
    """Generate text from the model. The splitter can disable the LoRA adapter."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    def run_generate():
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
            )
        return tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

    if use_base_model and hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            return run_generate()
    return run_generate()

def extract_json(raw_output):
    """Extract JSON even when the model adds text or markdown fences."""
    clean_json = raw_output.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(clean_json)
    except json.JSONDecodeError:
        pass

    if "{" in clean_json and "}" in clean_json:
        candidate = clean_json[clean_json.find("{"):clean_json.rfind("}") + 1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    return None

def ask_llm(system_prompt, user_content, max_tokens=150, return_raw=False, use_base_model=False):
    """Universal LLM call helper returning JSON plus optional raw output."""
    raw_output = generate_llm_output(system_prompt, user_content, max_tokens=max_tokens, use_base_model=use_base_model)
    payload = extract_json(raw_output)

    if return_raw:
        return payload, raw_output
    return payload

def parse_category(raw_output: str):
    """Extract the category from free-form output or messy JSON."""
    text = raw_output.strip()
    if "{" in text and "}" in text:
        candidate = text[text.find("{"):text.rfind("}") + 1]
        try:
            payload = json.loads(candidate)
            if isinstance(payload, dict):
                if "category" in payload:
                    return str(payload["category"]).strip()
                if "categoria" in payload:
                    return str(payload["categoria"]).strip()
                if "label" in payload:
                    return str(payload["label"]).strip()
        except json.JSONDecodeError:
            pass

    for category in MAPPING_9_CLASSES.keys():
        if category.lower() in text.lower():
            return category
    return None

def normalize_sub_questions(candidates, original_question):
    """Clean and limit sub-questions generated by rules or the LLM."""
    sub_questions = []
    for candidate in candidates or []:
        text = str(candidate).strip()
        text = re.sub(r"^Q\.\s*", "", text, flags=re.IGNORECASE).strip()
        text = re.sub(r"^(And|and)\s+", "", text).strip()
        starter_pattern = r"\b(do|does|did|is|are|was|were|can|could|will|would|should|how|why|what|when|where|if)\b"
        starter_matches = list(re.finditer(starter_pattern, text, flags=re.IGNORECASE))
        if starter_matches:
            last_sentence_break = max(text.rfind(". "), text.rfind("? "))
            viable_starters = [m for m in starter_matches if m.start() > last_sentence_break]
            starter = viable_starters[0] if viable_starters else starter_matches[0]
            text = text[starter.start():].strip()
        if text and text[0].islower():
            text = text[0].upper() + text[1:]
        if len(text.split()) < 3:
            continue
        if text not in sub_questions:
            sub_questions.append(text)

    return sub_questions[:3] if sub_questions else [original_question]

def rule_based_question_split(question):
    """Deterministic splitter for obvious cases: multiple '?' or interrogative connectors."""
    text = " ".join(str(question).split())
    if text.count("?") >= 2:
        candidates = re.findall(r"[^?]+\?", text)
        return normalize_sub_questions(candidates, question)

    connector_pattern = r"\s+(?=(?:And|and)\s+(?:do|does|did|is|are|was|were|can|could|will|would|should|how|why|what|when|where|if)\b)"
    candidates = re.split(connector_pattern, text)
    if len(candidates) > 1:
        return normalize_sub_questions(candidates, question)

    return [question]

def split_question(question):
    """Split a complex question into sub-questions and return debug metadata."""
    rule_split = rule_based_question_split(question)
    if len(rule_split) > 1:
        raw_output = "__RULE_BASED_SPLIT__"
        return rule_split, raw_output, "rule_based"

    if len(str(question).split()) < 8:
        raw_output = "__SKIPPED_SHORT_QUESTION__"
        return [question], raw_output, "short_question"

    result, raw_output = ask_llm(
        PROMPT_SPLITTER,
        f"Question to analyze: {question}",
        max_tokens=200,
        return_raw=True,
        use_base_model=True,
    )
    if result and "sub_questions" in result:
        sub_questions = normalize_sub_questions(result["sub_questions"], question)
        return sub_questions, raw_output, "llm_base"

    return [question], raw_output, "parser_fallback"

def evaluate_answer_pair(question_text, full_answer):
    """Evaluate an answer against one question using the Task 2 evaluator prompt."""
    user_content = f"Question: {question_text}\nAnswer: {full_answer}"

    def run_eval(prompt_override=None):
        system_prompt = prompt_override or PROMPT_EVALUATOR
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id
            )

        return tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

    raw_output = run_eval()
    raw_outputs = [raw_output]
    predicted_technique = parse_category(raw_output)

    if predicted_technique not in MAPPING_9_CLASSES:
        retry_prompt = PROMPT_EVALUATOR + "\nReply only with valid JSON."
        raw_output = run_eval(prompt_override=retry_prompt)
        raw_outputs.append(raw_output)
        predicted_technique = parse_category(raw_output)

    if predicted_technique not in MAPPING_9_CLASSES:
        predicted_technique = "Dodging"

    return predicted_technique, raw_outputs

def evaluate_sub_question(sub_question, full_answer):
    """Evaluate the answer against a single sub-question."""
    return evaluate_answer_pair(sub_question, full_answer)

def chaining_inference(example):
    """Run the full prompt-chaining pipeline and aggregate sub-question predictions."""
    question = example.get('interview_question', '')
    answer = example.get('interview_answer', '')

    sub_questions, raw_splitter_output, splitter_source = split_question(question)
    direct_technique, raw_direct_outputs = evaluate_answer_pair(question, answer)

    evaluations = []
    raw_evaluator_outputs = []
    for sq in sub_questions:
        evaluation, raw_outputs = evaluate_sub_question(sq, answer)
        evaluations.append(evaluation)
        raw_evaluator_outputs.append(raw_outputs)

    if not evaluations:
        evaluations = ["Dodging"]

    # Aggregation logic
    if len(sub_questions) == 1:
        final_technique = evaluations[0]
    else:
        counts = Counter(evaluations)
        n = len(evaluations)
        n_explicit = counts.get("Explicit", 0)
        n_nonreply = counts.get("Declining to answer", 0) + counts.get("Claims ignorance", 0)

        if n_explicit == n:
            final_technique = "Explicit"
        elif n >= 3 and n_explicit / n >= 0.67:
            final_technique = "Explicit"
        elif n_nonreply == n:
            final_technique = "Declining to answer" if counts.get("Declining to answer", 0) >= counts.get("Claims ignorance", 0) else "Claims ignorance"
        elif n_explicit > 0:
            final_technique = "Partial/half-answer"
        else:
            final_technique = counts.most_common(1)[0][0]

    # Safety fallback
    if final_technique not in MAPPING_9_CLASSES:
        final_technique = "Dodging"

    # Hybrid fallback: keep the decomposition signal, but recover clear direct predictions.
    if MAPPING_9_CLASSES[final_technique] == "Ambivalent" and MAPPING_9_CLASSES.get(direct_technique) in {"Clear Reply", "Clear Non-Reply"}:
        final_technique = direct_technique

    macro_clarity = MAPPING_9_CLASSES[final_technique]

    return final_technique, macro_clarity, sub_questions, evaluations, raw_splitter_output, splitter_source, raw_evaluator_outputs, direct_technique, raw_direct_outputs


## Pipeline debug sample

Inspect splitter behavior and sample predictions before running the full test set.


In [ ]:
import collections

N_DEBUG = min(30, len(test_dataset))
print(f"=== DEBUG: Analysis on {N_DEBUG} samples ===\n")

split_counts = []
sample_logs = []

for i in range(N_DEBUG):
    example = test_dataset[i]
    technique, clarity, sub_q, sub_evals, raw_splitter, splitter_source, raw_evaluator_outputs, direct_technique, raw_direct_outputs = chaining_inference(example)
    split_counts.append(len(sub_q))
    true_label = str(example.get('clarity_label', '')).strip()
    sample_logs.append({
        "true_label": true_label, "predicted": clarity,
        "technique": technique, "n_subs": len(sub_q), "splitter_source": splitter_source,
        "sub_evals": sub_evals, "direct_technique": direct_technique,
        "raw_evaluator_outputs": raw_evaluator_outputs, "raw_direct_outputs": raw_direct_outputs
    })

print("Sub-question count distribution per question:")
print(dict(collections.Counter(split_counts)))
pct_split = sum(1 for x in split_counts if x > 1) / len(split_counts) * 100
print(f"% questions with split: {pct_split:.1f}%\n")

print("Predicted technique distribution (sample):")
print(dict(collections.Counter(l['technique'] for l in sample_logs)))
print()

print("Splitter source (sample):")
print(dict(collections.Counter(l['splitter_source'] for l in sample_logs)))
print()

print("Examples (first 5):")
for l in sample_logs[:5]:
    print(f"  True: {l['true_label']:15s} | Pred: {l['predicted']:15s} ({l['technique']}) "
          f"| N_subs: {l['n_subs']} | Source: {l['splitter_source']} | Direct: {l['direct_technique']} | Evals: {l['sub_evals']}")


## Final evaluation

Evaluate prompt chaining on the full test set and save decomposition logs.


In [ ]:
y_true_clarity = []
y_pred_clarity = []
y_pred_technique = []
decomposition_logs = []

print(f"Starting Prompt Chaining on {len(test_dataset)} examples (this will take time)...")

# Use a smaller number for an initial test, e.g. range(10)
for i in tqdm(range(len(test_dataset))):
    example = test_dataset[i]
    y_true_clarity.append(str(example.get('clarity_label', '')).strip())

    technique, clarity, sub_q, sub_evals, raw_splitter, splitter_source, raw_evaluator_outputs, direct_technique, raw_direct_outputs = chaining_inference(example)
    
    y_pred_technique.append(technique)
    y_pred_clarity.append(clarity)
    decomposition_logs.append({"sub_questions": sub_q, "evaluations": sub_evals, "raw_splitter_output": raw_splitter, "splitter_source": splitter_source, "raw_evaluator_outputs": raw_evaluator_outputs, "direct_technique": direct_technique, "raw_direct_outputs": raw_direct_outputs})

# Report and plots
LABELS = ["Ambivalent", "Clear Reply", "Clear Non-Reply"]

print("\n" + "=" * 60)
print("FINAL REPORT — TASK 3 (PROMPT CHAINING)")
print("=" * 60)
print(classification_report(y_true_clarity, y_pred_clarity, labels=LABELS, digits=3))

os.makedirs(CHAINING_DIR, exist_ok=True)

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true_clarity, y_pred_clarity, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predictions (Prompt Chaining)')
plt.ylabel('True Labels')
plt.title('Confusion Matrix — Prompt Chaining Pipeline')
plt.tight_layout()

# Save the confusion matrix image safely
chaining_plot_path = os.path.join(CHAINING_DIR, "confusion_matrix_chaining.png")
plt.savefig(chaining_plot_path, bbox_inches='tight', dpi=300)
plt.show()

# Save advanced results to CSV
results_df = pd.DataFrame({
    'Original_Question': [ex.get('interview_question', '') for ex in test_dataset],
    'Politician_Answer': [ex.get('interview_answer', '') for ex in test_dataset],
    'Sub_Questions': [str(log["sub_questions"]) for log in decomposition_logs],
    'Raw_Splitter_Output': [log["raw_splitter_output"] for log in decomposition_logs],
    'Splitter_Source': [log["splitter_source"] for log in decomposition_logs],
    'Single_Evaluations': [str(log["evaluations"]) for log in decomposition_logs],
    'Raw_Evaluator_Outputs': [str(log["raw_evaluator_outputs"]) for log in decomposition_logs],
    'Direct_Technique': [log["direct_technique"] for log in decomposition_logs],
    'Raw_Direct_Outputs': [str(log["raw_direct_outputs"]) for log in decomposition_logs],
    'Aggregated_Technique': y_pred_technique,
    'Predicted_Macro_Clarity': y_pred_clarity,
    'True_Macro_Clarity': y_true_clarity
})

chaining_csv_path = os.path.join(CHAINING_DIR, "prompt_chaining_results.csv")
results_df.to_csv(chaining_csv_path, index=False)
print(f"CSV with decomposition logs saved to: {chaining_csv_path}")
